In [36]:
import faiss
import numpy as np
import pandas as pd
import os
from collections import Counter
from sentence_transformers import SentenceTransformer

In [37]:
df = pd.read_csv("data.csv")
meta_df = df.drop('casuality', axis=1)
meta_df = meta_df.rename(columns={"img": "image_filename", "neckline": "neck"})
meta_df = meta_df.reset_index().rename(columns={"index": "faiss_id"})
meta_df.reset_index(drop=True, inplace=True)

In [38]:
meta_df.head()

,faiss_id,image_filename,brand,sleeve,neck,primary_color,secondary_color,fit,pants_color,hair_color
0,0,00000_00.jpg,levi's,short sleeve,round neck,white,red,tight fit,blue,brown
1,1,00001_00.jpg,levi's,short sleeve,round neck,black,red,tight fit,black,brown
2,2,00002_00.jpg,unknown,long sleeve,round neck,black,unknown,tight fit,black,blonde
3,3,00003_00.jpg,unknown,short sleeve,round neck,blue,unknown,tight fit,black,black
4,4,00005_00.jpg,levi's,short sleeve,round neck,white,red,loose fit,red,blonde


In [39]:
model = SentenceTransformer("all-mpnet-base-v2")

In [40]:
text_fields = [
    "brand", "sleeve", "neck", "primary_color", "secondary_color",
    "fit", "pants_color", "hair_color"
]

In [41]:
field_indexes = {}
field_embeddings = {}

os.makedirs("faiss_data", exist_ok=True)

for field in text_fields:
    emb_path = f"faiss_data/{field}_embeddings.npy"
    idx_path = f"faiss_data/{field}.index"

    if os.path.exists(emb_path) and os.path.exists(idx_path):
        emb = np.load(emb_path)
        index = faiss.read_index(idx_path)
    else:
        corpus = meta_df[field].astype(str).tolist()
        emb = model.encode(corpus, convert_to_numpy=True, show_progress_bar=False)
        faiss.normalize_L2(emb)
        index = faiss.IndexFlatIP(emb.shape[1])
        index.add(emb)
        np.save(emb_path, emb)
        faiss.write_index(index, idx_path)

    field_indexes[field] = index
    field_embeddings[field] = emb

In [42]:
def compute_frequencies(items, fields):
    N = len(items)
    freq = {}
    for f in fields:
        cnt = Counter(item.get(f) for item in items)
        freq[f] = {v: cnt[v] / N for v in cnt}
    return freq

In [43]:
def recommend_clothes(
    meta_df,
    user_items,
    fields_to_match,
    alpha=0.5,
    beta=1.0,
    gamma=0.5,
    top_k=5
):
    all_fields = text_fields.copy()
    fields_to_novel = [f for f in all_fields if f not in fields_to_match]

    # Frequency profile for novelty penalty
    freq_profiles = compute_frequencies(user_items, fields_to_novel)

    scores = np.zeros(len(meta_df))

    last = user_items[-1]

    # Similarity for selected fields
    for f in fields_to_match:
        v = model.encode([last.get(f, "")], convert_to_numpy=True)
        faiss.normalize_L2(v)
        D, _ = field_indexes[f].search(v, len(meta_df))
        scores += D[0]

    # Novelty penalty
    if fields_to_novel and gamma > 0:
        for f in fields_to_novel:
            v = model.encode([last.get(f, "")], convert_to_numpy=True)
            faiss.normalize_L2(v)
            D, _ = field_indexes[f].search(v, len(meta_df))
            scores -= gamma * D[0]

    scores /= len(fields_to_match)

    # Prepare final scores
    results = []
    top_idx = np.argsort(scores)[::-1]

    for idx in top_idx:
        row = meta_df.iloc[idx]
        cand = {f: row.get(f) for f in text_fields}
        bonus = sum(1 for f in fields_to_match if cand[f] in {i.get(f) for i in user_items}) / len(fields_to_match) if fields_to_match else 0
        penalty = sum(freq_profiles[f].get(cand[f], 0.0) for f in fields_to_novel)
        final_score = float(scores[idx]) + alpha * bonus - beta * penalty
        results.append({
            "faiss_id": int(row.faiss_id),
            "image": row.image_filename,
            "score": final_score,
            "base_sim": float(scores[idx]),
            "bonus": bonus,
            "penalty": penalty,
            **cand
        })
        if len(results) >= top_k:
            break

    return sorted(results, key=lambda x: x["score"], reverse=True)


In [45]:
user_items = [
    {"brand":"levi's","sleeve":"short sleeve","neck":"round neck","primary_color":"white","secondary_color":"red","fit":"tight fit","pants_color":"blue","hair_color":"brown"},
    {"brand":"levi's","sleeve":"short sleeve","neck":"round neck","primary_color":"black","secondary_color":"red","fit":"tight fit","pants_color":"black","hair_color":"brown"}
]
fields_to_match = ["hair_color","fit"]
recs = recommend_clothes(
    meta_df=meta_df,
    user_items=user_items,
    fields_to_match=fields_to_match,
    alpha=0.6,
    beta=1.2,
    gamma=0.3,
    top_k=3
)
for r in recs:
    print(r)

{'faiss_id': 2827, 'image': '03689_00.jpg', 'score': -0.37701722581405184, 'base_sim': 0.5229827741859481, 'bonus': 0.5, 'penalty': 1.0, 'brand': 'max&co', 'sleeve': 'short sleeve', 'neck': 'boat neck', 'primary_color': 'red', 'secondary_color': 'unknown', 'fit': 'tight fit', 'pants_color': 'red', 'hair_color': 'black'}
{'faiss_id': 2825, 'image': '03687_00.jpg', 'score': -0.9906536621274424, 'base_sim': 0.5093463378725573, 'bonus': 0.5, 'penalty': 1.5, 'brand': 'unknown', 'sleeve': 'short sleeve', 'neck': 'hal', 'primary_color': 'beige', 'secondary_color': 'white', 'fit': 'tight fit', 'pants_color': 'blue', 'hair_color': 'black'}
{'faiss_id': 2826, 'image': '03688_00.jpg', 'score': -1.8819114190759136, 'base_sim': 0.5180885809240863, 'bonus': 0.0, 'penalty': 2.0, 'brand': 'roxy', 'sleeve': 'short sleeve', 'neck': 'scoop neck', 'primary_color': 'black', 'secondary_color': 'white', 'fit': 'loose fit', 'pants_color': 'black', 'hair_color': 'blonde'}
